In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import os

# Chemins
data_dir = r"C:\Users\MSI\Desktop\COVID19-XRay-Classification\data"
IMG_SIZE = (128, 128)  # Réduire la taille pour aller plus vite
BATCH_SIZE = 64
EPOCHS = 15

# Analyser le déséquilibre
temp_gen = ImageDataGenerator(rescale=1./255, validation_split=0.2).flow_from_directory(
    data_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=False
)

class_counts = np.bincount(temp_gen.classes)
class_names = list(temp_gen.class_indices.keys())

# Calcul des poids
class_weights = {}
for i in range(len(class_counts)):
    class_weights[i] = sum(class_counts) / (len(class_counts) * class_counts[i])

print("Class weights:", class_weights)

# Générateurs
train_datagen = ImageDataGenerator(
    rescale=1./255, validation_split=0.2,
    rotation_range=20, zoom_range=0.2,
    horizontal_flip=True
)

train_gen = train_datagen.flow_from_directory(
    data_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training'
)

val_gen = ImageDataGenerator(rescale=1./255, validation_split=0.2).flow_from_directory(
    data_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation'
)

# Modèle CNN simple
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dense(4, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Entraînement avec class_weight
history = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS, class_weight=class_weights,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
)

# Sauvegarde
save_dir = r"C:\Users\MSI\Desktop\COVID19-XRay-Classification\models"
os.makedirs(save_dir, exist_ok=True)
model.save(os.path.join(save_dir, 'simple_cnn_model.h5'))

Found 33866 images belonging to 4 classes.
Class weights: {0: 1.4632734185966125, 1: 0.8800935550935551, 2: 0.5191623742948246, 3: 3.9342472118959106}
Found 33866 images belonging to 4 classes.
Found 8464 images belonging to 4 classes.


C:\Users\MSI\anaconda3\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\MSI\anaconda3\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
530/530 ━━━━━━━━━━━━━━━━━━━━ 521s 975ms/step - accuracy: 0.4343 - loss: 1.1246 - val_accuracy: 0.6704 - val_loss: 0.8253
Epoch 2/15
530/530 ━━━━━━━━━━━━━━━━━━━━ 466s 879ms/step - accuracy: 0.5560 - loss: 0.8739 - val_accuracy: 0.6369 - val_loss: 0.8596
Epoch 3/15
530/530 ━━━━━━━━━━━━━━━━━━━━ 514s 969ms/step - accuracy: 0.5958 - loss: 0.7996 - val_accuracy: 0.6893 - val_loss: 0.6984
Epoch 4/15
530/530 ━━━━━━━━━━━━━━━━━━━━ 484s 913ms/step - accuracy: 0.6143 - loss: 0.7774 - val_accuracy: 0.7553 - val_loss: 0.6005
Epoch 5/15
530/530 ━━━━━━━━━━━━━━━━━━━━ 466s 879ms/step - accuracy: 0.6236 - loss: 0.7494 - val_accuracy: 0.7961 - val_loss: 0.5317
Epoch 6/15
530/530 ━━━━━━━━━━━━━━━━━━━━ 466s 878ms/step - accuracy: 0.6422 - loss: 0.7172 - val_accuracy: 0.7990 - val_loss: 0.5210
Epoch 7/15
530/530 ━━━━━━━━━━━━━━━━━━━━ 492s 927ms/step - accuracy: 0.6618 - loss: 0.6916 - val_accuracy: 0.8065 - val_loss: 0.5069
Epoch 8/15
530/530 ━━━━━━━━━━━━━━━━━━━━ 475s 895ms/step - accuracy: 0.6717 -